# Monte Carlo vs Actual Market Prices

Compares MC simulation estimates against **actual option prices** from Yahoo Finance.

## Approach
1. Load actual option prices from `default.actual_option_prices`
2. For each unique (ticker, K, T) combination, run MC simulations at multiple scales
3. Compare MC call/put prices to market mid-prices
4. Analyze accuracy: MAE, MAPE, correlation

## Configuration
- **Alt weight:** 0.1 (smallest, most conservative)
- **Run scales:** 10K, 100K, 500K, 1M, 2M
- **Methods:** All 6 (historical, window, window_10d, window_20d, student_t, black_scholes)
- **Comparison metric:** Mid-price (bid+ask)/2 as ground truth

In [0]:
%pip install scipy yfinance --quiet
dbutils.library.restartPython()

In [0]:
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

src_dir = "/Workspace/Users/spyros.louvis@ctpsandbox.com/pub-python-repo/databricks/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from config.settings import FULL_TABLE_NAME, get_logger
from utils.simulation_helpers import (
    load_historical_data, fit_distributions, run_simulations, get_risk_free_rate,
)

logger = get_logger("mc_vs_actual_test")
logger.info("Modules loaded.")

In [0]:
# ---------------------------------------------------------------------------
# Test configuration
# ---------------------------------------------------------------------------
ACTUAL_PRICES_TABLE = "default.actual_option_prices"
HISTORICAL_TABLE = FULL_TABLE_NAME

# Simulation parameters
ALT_WEIGHT = 0.1  # smallest alt_weight
RUN_SCALES = [10_000, 100_000, 500_000, 1_000_000, 2_000_000]

# Max unique (ticker, K, T) combos to test per ticker (to keep runtime reasonable)
MAX_COMBOS_PER_TICKER = 15

logger.info(f"Alt weight: {ALT_WEIGHT}")
logger.info(f"Run scales: {RUN_SCALES}")
logger.info(f"Max combos per ticker: {MAX_COMBOS_PER_TICKER}")

In [0]:
# ---------------------------------------------------------------------------
# Load actual option prices
# ---------------------------------------------------------------------------
actual_df = spark.table(ACTUAL_PRICES_TABLE).toPandas()

print(f"Loaded {len(actual_df)} actual option prices")
print(f"Tickers: {sorted(actual_df['ticker'].unique())}")
print(f"Option types: {actual_df['option_type'].value_counts().to_dict()}")
print(f"Days to expiry: {actual_df['days_to_expiry'].min()} - {actual_df['days_to_expiry'].max()}")
print(f"\nPer ticker:")
print(actual_df.groupby('ticker').agg(
    rows=('strike', 'count'),
    S0=('underlying_price', 'first'),
    expirations=('days_to_expiry', 'nunique'),
).to_string())

In [0]:
# ---------------------------------------------------------------------------
# For each ticker: load historical data, fit distributions, run simulations
# Compare MC prices to actual market prices at the highest scale
# ---------------------------------------------------------------------------
r = get_risk_free_rate()

tickers_in_actual = actual_df["ticker"].unique()

# Map from option ticker to historical ticker (SPY -> ^GSPC for S&P500)
TICKER_MAP = {
    "SPY": "^GSPC",  # SPY tracks S&P 500 (same log returns, different price level)
    "AAPL": "AAPL",
    "MSFT": "MSFT",
    "GOOGL": "GOOGL",
    "AMZN": "AMZN",
    "META": "META",
}

all_comparison_rows = []

for opt_ticker in tickers_in_actual:
    hist_ticker = TICKER_MAP.get(opt_ticker, opt_ticker)
    print(f"\n{'#'*60}")
    print(f"  TICKER: {opt_ticker} (historical: {hist_ticker})")
    print(f"{'#'*60}")
    
    # Load historical data for this ticker (log returns + S0)
    try:
        log_returns, _ = load_historical_data(spark, HISTORICAL_TABLE, hist_ticker)
        dist_params = fit_distributions(log_returns)
    except Exception as e:
        print(f"  [SKIP] No historical data for {hist_ticker}: {e}")
        continue
    
    # Use actual underlying price from options data (fixes SPY/^GSPC scale mismatch)
    ticker_actual = actual_df[actual_df["ticker"] == opt_ticker].copy()
    S0 = float(ticker_actual["underlying_price"].iloc[0])
    
    # Get unique (strike, days_to_expiry, option_type) combos, sample if too many
    combos = ticker_actual.groupby(["strike", "days_to_expiry", "option_type"]).first().reset_index()
    if len(combos) > MAX_COMBOS_PER_TICKER * 2:  # *2 for call+put
        combos = combos.sample(n=MAX_COMBOS_PER_TICKER * 2, random_state=42)
    
    # Get unique (K, T) pairs
    kt_pairs = combos[["strike", "days_to_expiry"]].drop_duplicates()
    
    print(f"  S0=${S0:.2f}, vol={dist_params['vol']:.4f}, {len(kt_pairs)} (K,T) pairs")
    
    for _, kt_row in kt_pairs.iterrows():
        K = float(kt_row["strike"])
        T = int(kt_row["days_to_expiry"])
        
        # Run MC simulation at each scale
        for num_runs in RUN_SCALES:
            results_pdf = run_simulations(
                log_returns, S0, K, T, num_runs,
                dist_params=dist_params,
                alt_weight=ALT_WEIGHT,
                r=r,
            )
            
            # Get actual prices for this (ticker, K, T)
            mask = (
                (ticker_actual["strike"] == K) &
                (ticker_actual["days_to_expiry"] == T)
            )
            actual_subset = ticker_actual[mask]
            
            actual_call = actual_subset[actual_subset["option_type"] == "call"]["mid_price"].values
            actual_put = actual_subset[actual_subset["option_type"] == "put"]["mid_price"].values
            actual_call_price = float(actual_call[0]) if len(actual_call) > 0 else None
            actual_put_price = float(actual_put[0]) if len(actual_put) > 0 else None
            
            for _, sim_row in results_pdf.iterrows():
                row = {
                    "ticker": opt_ticker,
                    "K": K,
                    "T": T,
                    "S0": S0,
                    "num_runs": num_runs,
                    "method": sim_row["method"],
                    "mc_call": sim_row["CallPrice"],
                    "mc_put": sim_row["PutPrice"],
                    "actual_call": actual_call_price,
                    "actual_put": actual_put_price,
                }
                all_comparison_rows.append(row)
    
    print(f"  [OK] {len(kt_pairs)} pairs x {len(RUN_SCALES)} scales x 6 methods = {len(kt_pairs)*len(RUN_SCALES)*6} rows")

comparison_df = pd.DataFrame(all_comparison_rows)
print(f"\n{'='*60}")
print(f"TOTAL COMPARISON ROWS: {len(comparison_df)}")
print(f"{'='*60}")

In [0]:
# ---------------------------------------------------------------------------
# Accuracy analysis at highest scale
# ---------------------------------------------------------------------------
high_scale = RUN_SCALES[-1]
high_df = comparison_df[comparison_df["num_runs"] == high_scale].copy()

# Call accuracy (where actual is available)
call_df = high_df[high_df["actual_call"].notna() & (high_df["actual_call"] > 0.5)].copy()
call_df["call_error"] = call_df["mc_call"] - call_df["actual_call"]
call_df["call_pct_error"] = (call_df["call_error"] / call_df["actual_call"] * 100)

# Put accuracy
put_df = high_df[high_df["actual_put"].notna() & (high_df["actual_put"] > 0.5)].copy()
put_df["put_error"] = put_df["mc_put"] - put_df["actual_put"]
put_df["put_pct_error"] = (put_df["put_error"] / put_df["actual_put"] * 100)

print(f"{'='*70}")
print(f"ACCURACY REPORT ({high_scale:,} runs, alt_weight={ALT_WEIGHT})")
print(f"{'='*70}")

print(f"\n  CALL OPTIONS ({len(call_df)} observations):")
for method in call_df["method"].unique():
    m = call_df[call_df["method"] == method]
    mae = m["call_error"].abs().mean()
    mape = m["call_pct_error"].abs().mean()
    median_err = m["call_pct_error"].median()
    print(f"    {method:<14s}  MAE=${mae:>7.2f}  MAPE={mape:>6.1f}%  Median={median_err:>+6.1f}%")

print(f"\n  PUT OPTIONS ({len(put_df)} observations):")
for method in put_df["method"].unique():
    m = put_df[put_df["method"] == method]
    mae = m["put_error"].abs().mean()
    mape = m["put_pct_error"].abs().mean()
    median_err = m["put_pct_error"].median()
    print(f"    {method:<14s}  MAE=${mae:>7.2f}  MAPE={mape:>6.1f}%  Median={median_err:>+6.1f}%")

# Best method overall
print(f"\n  BEST METHOD (lowest MAPE):")
call_mapes = call_df.groupby("method")["call_pct_error"].apply(lambda x: x.abs().mean())
put_mapes = put_df.groupby("method")["put_pct_error"].apply(lambda x: x.abs().mean())
print(f"    Calls: {call_mapes.idxmin()} ({call_mapes.min():.1f}%)")
print(f"    Puts:  {put_mapes.idxmin()} ({put_mapes.min():.1f}%)")

In [0]:
# ---------------------------------------------------------------------------
# Visualization: MC vs Actual prices
# ---------------------------------------------------------------------------
high_scale = RUN_SCALES[-1]
high_df = comparison_df[comparison_df["num_runs"] == high_scale].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Top Left: Scatter plot MC vs Actual (Calls) ---
ax = axes[0, 0]
call_df = high_df[high_df["actual_call"].notna() & (high_df["actual_call"] > 0.5)].copy()

for method in ["black_scholes", "historical", "student_t"]:
    m = call_df[call_df["method"] == method]
    ax.scatter(m["actual_call"], m["mc_call"], alpha=0.5, s=20, label=method)

# Perfect prediction line
max_val = max(call_df["actual_call"].max(), call_df["mc_call"].max())
ax.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="Perfect")
ax.set_xlabel("Actual Call Price ($)")
ax.set_ylabel("MC Call Price ($)")
ax.set_title(f"MC vs Actual: Call Prices ({high_scale:,} runs)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Top Right: Scatter plot MC vs Actual (Puts) ---
ax = axes[0, 1]
put_df = high_df[high_df["actual_put"].notna() & (high_df["actual_put"] > 0.5)].copy()

for method in ["black_scholes", "historical", "student_t"]:
    m = put_df[put_df["method"] == method]
    ax.scatter(m["actual_put"], m["mc_put"], alpha=0.5, s=20, label=method)

max_val = max(put_df["actual_put"].max(), put_df["mc_put"].max()) if not put_df.empty else 100
ax.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="Perfect")
ax.set_xlabel("Actual Put Price ($)")
ax.set_ylabel("MC Put Price ($)")
ax.set_title(f"MC vs Actual: Put Prices ({high_scale:,} runs)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Bottom Left: MAPE by method (bar chart) ---
ax = axes[1, 0]
call_df_all = high_df[high_df["actual_call"].notna() & (high_df["actual_call"] > 0.5)].copy()
call_df_all["call_pct_error"] = ((call_df_all["mc_call"] - call_df_all["actual_call"]) / call_df_all["actual_call"] * 100).abs()

mape_by_method = call_df_all.groupby("method")["call_pct_error"].mean().sort_values()
colors = ["#2ecc71" if m == "black_scholes" else "#3498db" for m in mape_by_method.index]
ax.barh(mape_by_method.index, mape_by_method.values, color=colors)
ax.set_xlabel("MAPE (%)")
ax.set_title("Call Price MAPE by Method")
ax.grid(axis="x", alpha=0.3)
for i, (method, val) in enumerate(mape_by_method.items()):
    ax.text(val + 0.5, i, f"{val:.1f}%", va="center", fontsize=9)

# --- Bottom Right: Error by scale (convergence to actual) ---
ax = axes[1, 1]
call_all = comparison_df[comparison_df["actual_call"].notna() & (comparison_df["actual_call"] > 0.5)].copy()
call_all["abs_pct_error"] = ((call_all["mc_call"] - call_all["actual_call"]) / call_all["actual_call"] * 100).abs()

for method in ["black_scholes", "historical", "window_20d", "student_t"]:
    m = call_all[call_all["method"] == method]
    scale_mape = m.groupby("num_runs")["abs_pct_error"].mean()
    ax.plot(scale_mape.index, scale_mape.values, marker="o", label=method, linewidth=1.5)

ax.set_xscale("log")
ax.set_xlabel("Number of MC Runs (log)")
ax.set_ylabel("MAPE vs Actual (%)")
ax.set_title("Error Convergence to Actual Prices")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/mc_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /tmp/mc_vs_actual.png")

In [0]:
# ---------------------------------------------------------------------------
# ATM OPTIONS ONLY: filter to strikes within 5% of underlying price
# ---------------------------------------------------------------------------
ATM_THRESHOLD = 0.05  # |K/S0 - 1| <= 5%

high_scale = RUN_SCALES[-1]
high_df = comparison_df[comparison_df["num_runs"] == high_scale].copy()
high_df["moneyness"] = (high_df["K"] / high_df["S0"] - 1).abs()

atm_df = high_df[high_df["moneyness"] <= ATM_THRESHOLD].copy()

print(f"{'='*70}")
print(f"ATM-ONLY ACCURACY REPORT ({high_scale:,} runs, alt_weight={ALT_WEIGHT})")
print(f"Filter: |K/S0 - 1| <= {ATM_THRESHOLD*100:.0f}%")
print(f"{'='*70}")
print(f"\n  Total ATM observations: {len(atm_df)} (vs {len(high_df)} total)")
print(f"  Unique (ticker, K, T) combos: {atm_df[['ticker','K','T']].drop_duplicates().shape[0]}")

# Call accuracy (ATM)
call_atm = atm_df[atm_df["actual_call"].notna() & (atm_df["actual_call"] > 0.5)].copy()
call_atm["call_error"] = call_atm["mc_call"] - call_atm["actual_call"]
call_atm["call_pct_error"] = (call_atm["call_error"] / call_atm["actual_call"] * 100)

# Put accuracy (ATM)
put_atm = atm_df[atm_df["actual_put"].notna() & (atm_df["actual_put"] > 0.5)].copy()
put_atm["put_error"] = put_atm["mc_put"] - put_atm["actual_put"]
put_atm["put_pct_error"] = (put_atm["put_error"] / put_atm["actual_put"] * 100)

print(f"\n  ATM CALL OPTIONS ({len(call_atm)} observations):")
for method in sorted(call_atm["method"].unique()):
    m = call_atm[call_atm["method"] == method]
    mae = m["call_error"].abs().mean()
    mape = m["call_pct_error"].abs().mean()
    median_err = m["call_pct_error"].median()
    print(f"    {method:<14s}  MAE=${mae:>7.2f}  MAPE={mape:>6.1f}%  Median={median_err:>+6.1f}%")

print(f"\n  ATM PUT OPTIONS ({len(put_atm)} observations):")
for method in sorted(put_atm["method"].unique()):
    m = put_atm[put_atm["method"] == method]
    mae = m["put_error"].abs().mean()
    mape = m["put_pct_error"].abs().mean()
    median_err = m["put_pct_error"].median()
    print(f"    {method:<14s}  MAE=${mae:>7.2f}  MAPE={mape:>6.1f}%  Median={median_err:>+6.1f}%")

# Best method for ATM
call_mapes_atm = call_atm.groupby("method")["call_pct_error"].apply(lambda x: x.abs().mean())
put_mapes_atm = put_atm.groupby("method")["put_pct_error"].apply(lambda x: x.abs().mean())
print(f"\n  BEST ATM METHOD (lowest MAPE):")
print(f"    Calls: {call_mapes_atm.idxmin()} ({call_mapes_atm.min():.1f}%)")
print(f"    Puts:  {put_mapes_atm.idxmin()} ({put_mapes_atm.min():.1f}%)")
print(f"\n  IMPROVEMENT vs ALL strikes:")
print(f"    Calls: MAPE dropped from 46.4% (all) to {call_mapes_atm.min():.1f}% (ATM)")
print(f"    Puts:  MAPE dropped from 50.0% (all) to {put_mapes_atm.min():.1f}% (ATM)")

In [0]:
# ---------------------------------------------------------------------------
# ATM-only visualization
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Scatter: MC vs Actual Calls (ATM) ---
ax = axes[0]
for method in ["black_scholes", "historical", "window", "student_t"]:
    m = call_atm[call_atm["method"] == method]
    ax.scatter(m["actual_call"], m["mc_call"], alpha=0.6, s=30, label=method)

max_val = max(call_atm["actual_call"].max(), call_atm["mc_call"].max())
ax.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="Perfect")
ax.set_xlabel("Actual Call Price ($)")
ax.set_ylabel("MC Call Price ($)")
ax.set_title(f"ATM Calls: MC vs Actual ({high_scale:,} runs)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- MAPE bar chart (ATM calls) ---
ax = axes[1]
mape_sorted = call_mapes_atm.sort_values()
colors = ["#2ecc71" if m == mape_sorted.idxmin() else "#3498db" for m in mape_sorted.index]
ax.barh(mape_sorted.index, mape_sorted.values, color=colors)
ax.set_xlabel("MAPE (%)")
ax.set_title("ATM Call MAPE by Method")
ax.grid(axis="x", alpha=0.3)
for i, (method, val) in enumerate(mape_sorted.items()):
    ax.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=9)

# --- MAPE bar chart (ATM puts) ---
ax = axes[2]
mape_sorted_put = put_mapes_atm.sort_values()
colors_put = ["#2ecc71" if m == mape_sorted_put.idxmin() else "#e67e22" for m in mape_sorted_put.index]
ax.barh(mape_sorted_put.index, mape_sorted_put.values, color=colors_put)
ax.set_xlabel("MAPE (%)")
ax.set_title("ATM Put MAPE by Method")
ax.grid(axis="x", alpha=0.3)
for i, (method, val) in enumerate(mape_sorted_put.items()):
    ax.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/mc_vs_actual_atm.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"ATM chart saved. Showing {len(call_atm)} call + {len(put_atm)} put observations within {ATM_THRESHOLD*100:.0f}% of S0.")